<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/6-sub-agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Sub-Agents: Building Modular AI Workflows

This notebook demonstrates how to use Vectara's **sub-agents** capability to build modular, specialized AI workflows. Sub-agents allow a parent agent to delegate tasks to specialized child agents, enabling:

- **Context isolation**: Each sub-agent maintains its own conversation history
- **Specialized configuration**: Each sub-agent can have distinct instructions and tools
- **Reusability**: Build once, invoke from any parent agent
- **Parallel execution**: Run multiple sub-agents simultaneously
- **Better performance**: Smaller, focused agents make fewer mistakes

### Why Sub-Agents?

When agents face complex, multi-step tasks, they often run into context window limits or need specialized capabilities. Consider a comprehensive research assistant that needs to:

1. Analyze academic papers for theoretical foundations
2. Search product documentation for implementation details
3. Synthesize findings into actionable recommendations

A single monolithic agent trying to handle all of this might:
- Become confused between different instruction sets
- Consume excessive context with domain-specific guidelines
- Produce lower quality results due to competing priorities

**Sub-agents solve this by delegation**: the parent agent orchestrates, while specialized sub-agents focus on their domains.

We also demonstrate combining sub-agents with a **Lambda Tool** (API Payload Validator) that validates generated API code to catch hallucinations and ensure correctness before presenting to users.

## Getting Started

This notebook assumes you've completed Notebooks 1-5:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs)
- Notebook 2: Ingested AI research papers and Vectara documentation
- Notebook 3: Deleted documents from a corpus
- Notebook 4: Queried the data with various techniques
- Notebook 5: Created agents that can search and reason across data

Now we'll create a multi-agent system where specialized sub-agents handle domain-specific tasks.

## Setup

In [1]:
import os
import requests
import json
import time
from datetime import datetime

# Get credentials from environment variables
api_key = os.environ['VECTARA_API_KEY']

# Corpus keys from previous notebooks
research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

# Base API URL
BASE_URL = "https://api.vectara.io/v2"

# Common headers
headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

print(f"Research Corpus: {research_corpus_key}")
print(f"Docs Corpus: {docs_corpus_key}")

Research Corpus: tutorial-ai-research-papers
Docs Corpus: tutorial-vectara-docs


In [2]:
# Load the shared helpers (delete_and_create_agent / delete_and_create_tool).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import (
        delete_and_create_agent,
        delete_and_create_tool,
        find_agents_by_name,
    )
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import (
        delete_and_create_agent,
        delete_and_create_tool,
        find_agents_by_name,
    )

import vectara_utils
vectara_utils.configure(BASE_URL, headers)


## Step 1: Create Specialized Sub-Agents

We'll create three specialized agents that will serve as sub-agents:

1. **Research Paper Analyst**: Expert at analyzing academic papers on RAG, embeddings, and retrieval
2. **Documentation Expert**: Expert at finding implementation guidance from Vectara docs
3. **Web Search Expert**: Expert at searching the web for current information and news

Each agent has focused instructions and tools optimized for its domain.

### Sub-Agent 1: Research Paper Analyst

In [3]:
# Create Research Paper Analyst sub-agent
reranker_config = {
    "type": "chain",
    "rerankers": [
        {
            "type": "customer_reranker",
            "reranker_name": "qwen3-reranker",
            "limit": 25,
        },
        {
            "type": "mmr",
            "diversity_bias": 0.05
        }
    ],
}

research_analyst_config = {
    "name": "Research Paper Analyst",
    "description": "Specialized agent for analyzing academic research papers on RAG, embeddings, and retrieval techniques",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
            {
                "type": "inline",
                "name": "research_analyst_instructions",
                "template": """You are an expert academic research analyst specializing in AI, machine learning, and natural language processing.

Your expertise includes:
- Retrieval Augmented Generation (RAG) architectures
- Dense and sparse retrieval methods
- Embedding models and vector representations
- Transformer architectures and attention mechanisms
- Information retrieval benchmarks and evaluation metrics

When analyzing research papers:
1. Identify the key contributions and novel techniques
2. Explain technical concepts clearly with examples
3. Highlight practical implications and limitations
4. Compare with related work when relevant
5. Provide citations to the source papers

Always use tools to retrieve relevant content to answer user queries.
Always ground your response in the retrieved content. 
If you cannot answer the user question from the retrieved content from tools, just say "I don't know"

IMPORTANT: When responding, provide a complete, self-contained summary that includes all relevant findings."""
            }
        ],
            "output_parser": {"type": "default"}
        }
    },
    "tool_configurations": {
        "research_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": research_corpus_key}],
                    "limit": 100,
                    "context_configuration": {
                        "sentences_before": 2,
                        "sentences_after": 2
                    },
                    "reranker": reranker_config,                   
                },
                "save_history": True,
            }
        }
    }
}

research_analyst_key = delete_and_create_agent(research_analyst_config, "Research Paper Analyst")

Deleted existing agent 'Research Paper Analyst' (agt_research_paper_analyst_f0ec)


Created agent 'Research Paper Analyst' (key: agt_research_paper_analyst_8b79)


### Sub-Agent 2: Vectara Documentation Expert

In [4]:
# Create Documentation Expert sub-agent

docs_expert_config = {
    "name": "Documentation Expert",
    "description": "Specialized agent for finding implementation guidance and best practices from Vectara documentation",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
            {
                "type": "inline",
                "name": "docs_expert_instructions",
                "template": """You are a Vectara platform expert who helps developers implement AI solutions.

Your expertise includes:
- Vectara API integration (indexing, querying, agents)
- Corpus management and configuration
- Search optimization (hybrid search, reranking, filters)
- RAG implementation best practices
- SDK usage and code examples

When providing guidance:
1. Give specific, actionable implementation steps using the API.
2. Include relevant API endpoints and parameters
3. Your examples should show how to use the API, not using Vectara SDK.
4. Highlight configuration options and trade-offs
5. Point to relevant documentation sections
6. ALWAYS use the v2 API. URLs must use `https://api.vectara.io/v2/...` and payloads must follow the v2 shape (top-level `query`/`search`/`generation` — NOT wrapped in an outer `query` object). The `/v1/` API is deprecated; never produce v1 URLs or v1-shape payloads.

Always use tools to retrieve relevant content to answer user queries.
Always ground your response in the retrieved content. 
If you cannot answer the user question from the retrieved content from tools, just say "I don't know"

When responding, provide a complete, self-contained answer with all implementation details."""
            }
        ],
            "output_parser": {"type": "default"}
        }
    },
    "tool_configurations": {
        "docs_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": docs_corpus_key}],
                    "limit": 100,
                    "context_configuration": {
                        "sentences_before": 2,
                        "sentences_after": 2
                    },
                    "reranker": reranker_config,                   
                },
                "save_history": True,
            }
        }
    }
}

docs_expert_key = delete_and_create_agent(docs_expert_config, "Documentation Expert")

Deleted existing agent 'Documentation Expert' (agt_documentation_expert_48ce)


Created agent 'Documentation Expert' (key: agt_documentation_expert_716a)


### Sub-Agent 3: Web Search Expert

In [5]:
# Create Web Search Expert sub-agent

web_search_expert_config = {
    "name": "Web Search Expert",
    "description": "Specialized agent for searching the web for current information, news, and general knowledge",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
            {
                "type": "inline",
                "name": "web_search_expert_instructions",
                "template": """You are a web search expert who helps find current and relevant information from the internet.

Your expertise includes:
- Finding up-to-date information on any topic
- Researching current events and news
- Locating authoritative sources and references
- Comparing information across multiple sources
- Fact-checking and verification

When searching and responding:
1. Use web search to find relevant, current information
2. Prioritize authoritative and credible sources
3. Provide context about when information was published
4. Cite your sources with URLs when available
5. Synthesize information from multiple sources when appropriate

Always use the web search tool to find information.
If you cannot find relevant information, say so clearly.

IMPORTANT: When responding, provide a complete, well-sourced answer with citations."""
            }
        ],
            "output_parser": {"type": "default"}
        }
    },
    "tool_configurations": {
        "web_search": {
            "type": "web_search"
        }
    }
}

web_search_expert_key = delete_and_create_agent(web_search_expert_config, "Web Search Expert")

Deleted existing agent 'Web Search Expert' (agt_web_search_expert_b157)


Created agent 'Web Search Expert' (key: agt_web_search_expert_84fc)


## Step 2: Create an API Validator Lambda Tool

Before creating the orchestrator, we'll add a **Lambda Tool** that validates Vectara API payloads. This tool is *essential* for answering implementation questions because:

- LLMs may hallucinate field names (e.g., `corpus_id` instead of `corpus_key`)
- Parameter ranges need validation (e.g., `lexical_interpolation` must be 0-1)
- Required fields must be present for the code to actually work
- The orchestrator can't claim code is "working" without validating it first

Unlike optional analysis tools that LLMs may skip, this validator is essential—the orchestrator must use it before presenting API code examples to ensure correctness.

In [6]:
# Cleanup: any orchestrator from a prior run still references this tool,
# which blocks its DELETE with "tool is being used in different configurations".
# Tear down the orchestrator first, then wait for the async DELETE to clear
# before (re)creating the validator tool below.
for existing in find_agents_by_name("AI Research Orchestrator"):
    requests.delete(f"{BASE_URL}/agents/{existing['key']}", headers=headers)
    print(f"Deleted orchestrator agent: {existing['key']}")

# Wait for the async DELETE to clear before recreating the validator tool.
vectara_utils.wait_for_agent_gone("AI Research Orchestrator")

# Create the Vectara API Payload Validator lambda tool
api_validator_code = '''
import json
from typing import Literal

def process(
    endpoint: Literal["query", "index", "corpus", "agents"],
    payload: str,
    method: str = "POST"
) -> dict:
    """
    Validate a Vectara API request payload.
    
    Args:
        endpoint: Which v2 request the payload is for: query (POST /v2/query),
            index (POST /v2/corpora/{corpus_key}/documents), corpus (POST /v2/corpora),
            or agents (POST /v2/agents)
        payload: JSON string of the request body
        method: HTTP method (POST, GET, etc.)
    
    Returns:
        Validation result with errors and warnings
    """
    errors = []
    warnings = []
    
    # Parse the payload
    try:
        data = json.loads(payload)
    except json.JSONDecodeError as e:
        return {
            "valid": False,
            "endpoint": endpoint,
            "errors": [f"Invalid JSON: {str(e)}"],
            "warnings": []
        }
    
    # Validation rules by endpoint
    if endpoint == "query":
        # Required fields
        if "query" not in data:
            errors.append("Missing required field: 'query'")
        elif not isinstance(data["query"], str):
            errors.append("'query' must be the query text as a string, not an object")
        
        if "search" not in data:
            errors.append("Missing required field: 'search'")
        else:
            search = data.get("search", {})
            if "corpora" not in search:
                errors.append("Missing required field: 'search.corpora'")
            else:
                for i, corpus in enumerate(search.get("corpora", [])):
                    if "corpus_key" not in corpus:
                        errors.append(f"Missing corpus_key in corpora[{i}]")
                    
                    # Validate lexical_interpolation range
                    if "lexical_interpolation" in corpus:
                        li = corpus["lexical_interpolation"]
                        if not isinstance(li, (int, float)) or li < 0 or li > 1:
                            errors.append(f"lexical_interpolation must be 0-1, got: {li}")
            
            for field in search:
                if field not in {"corpora", "offset", "limit", "context_configuration", "reranker", "max_by"}:
                    errors.append(f"Invalid field 'search.{field}' - this is a hallucination")

            # Validate limit
            if "limit" in search:
                limit = search["limit"]
                if not isinstance(limit, int) or limit < 1:
                    warnings.append(f"limit should be positive integer, got: {limit}")
        
        # Only these top-level fields exist in a v2 query request; anything else is a hallucination
        allowed = {"query", "search", "generation", "save_history", "stream_response", "intelligent_query_rewriting"}
        hints = {"corpus_id": "search.corpora[].corpus_key",
                 "customer_id": "(not needed with API key)",
                 "num_results": "search.limit",
                 "query_text": "query",
                 "reranker": "search.reranker",
                 "context_configuration": "search.context_configuration",
                 "metadata_filter": "search.corpora[].metadata_filter"}
        for field in data:
            if field not in allowed:
                errors.append(f"Invalid field '{field}' - this is a hallucination. Did you mean: {hints.get(field, 'check docs')}")
    
    elif endpoint == "index":
        if "id" not in data:
            errors.append("Missing required field: 'id'")
        doc_type = data.get("type")
        if doc_type == "core":
            if "document_parts" not in data:
                errors.append("Core documents need 'document_parts': [{'text': ...}, ...]")
        elif doc_type == "structured":
            if "sections" not in data:
                errors.append("Structured documents need 'sections': [{'text': ...}, ...]")
        else:
            errors.append(f"Field 'type' must be 'core' or 'structured', got: {doc_type}")
        for field, fix in {"document_id": "id", "parts": "document_parts", "content": "document_parts"}.items():
            if field in data:
                errors.append(f"Invalid field '{field}' - use '{fix}'")
    
    elif endpoint == "agents":
        for field in ("name", "model", "tool_configurations"):
            if field not in data:
                errors.append(f"Missing required field: '{field}'")
        if "first_step_name" not in data and "first_step" not in data:
            errors.append("Missing entry point: set 'first_step_name' to a key of 'steps'")
    
    elif endpoint == "corpus":
        if "key" not in data:
            errors.append("Missing required field: 'key'")
    
    return {
        "valid": len(errors) == 0,
        "endpoint": endpoint,
        "method": method,
        "errors": errors,
        "warnings": warnings
    }
'''

api_validator_config = {
    "type": "lambda",
    "language": "python",
    "name": "vectara_api_validator",
    "title": "Vectara API Payload Validator",
    "description": "Validate Vectara API request payloads before presenting to users. Checks required fields, validates parameter ranges (e.g., lexical_interpolation 0-1), and catches common LLM hallucinations (wrong field names). Returns validation status and errors; if invalid, fix the payload and validate it again. ALWAYS use this before showing API code examples.",
    "code": api_validator_code
}

api_validator_id = delete_and_create_tool(api_validator_config, "vectara_api_validator")


Deleted orchestrator agent: agt_ai_research_orchestrator_ba17


Deleted existing tool 'vectara_api_validator' (tol_39970)


Created tool 'vectara_api_validator' (id: tol_39988)


## Step 3: Create the Parent Orchestrator Agent

Now we'll create a parent agent that can delegate to the sub-agents and use the API validator. The parent agent:
- Analyzes user requests to determine which sub-agent(s) to invoke
- Delegates domain-specific tasks to the appropriate sub-agent
- **Uses the API validator** to ensure generated code is correct before presenting it back to the user.
- Synthesizes responses from multiple sub-agents into a cohesive answer

### Sub-Agent Tool Configuration

Sub-agents are configured as tools using the `sub_agent` type:

```json
{
  "type": "sub_agent",
  "description_template": "Description the LLM sees when deciding to use this tool",
  "sub_agent_configuration": {
    "agent_key": "the_sub_agent_key",
    "session_mode": "ephemeral"
  }
}
```

### Session Modes

When specifying in the parent agent how sub-agents are to be used, you set the `session_mode`:
- **`ephemeral`** (default): Creates a new sub-agent session on every invocation and never resumes one (used below)
- **`persistent`**: Reuses one sub-agent session per (parent session, tool configuration) pair, created on the first invocation
- **`session_scoped`**: Resumes the sub-agent session named by `session_key` if one is provided, otherwise creates a new one. Only the parent session that created the named session may resume it
- **`agent_scoped`**: Like `session_scoped`, but any session of the creating parent agent may resume the named session
- **`llm_controlled`**: Deprecated alias for `session_scoped`


In [7]:
# Create the Orchestrator Agent with sub-agent tools and API validator
orchestrator_config = {
    "name": "AI Research Orchestrator",
    "description": "Orchestrator agent that delegates to specialized sub-agents for comprehensive AI research assistance",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
            {
                "type": "inline",
                "name": "orchestrator_instructions",
                "template": """You are an AI research orchestrator that helps users understand and implement AI technologies.

Your workflow:
1. Analyze the user's question to determine what expertise is needed.
2. Use appropriate sub-agent(s) to gather information:
   - research_analyst: for academic/theoretical content
   - docs_expert: for implementation guidance
   - web_search_expert: for current news/trends
3. When your response includes Vectara API code examples or implementation details:
   - Generate the JSON payload
   - ALWAYS validate it using the api_validator tool before presenting
   - If validation fails, fix the errors and validate again
   - Only present a payload after api_validator has returned valid=true for that exact payload
4. Synthesize all responses into a comprehensive answer.

IMPORTANT: 
- Never show API code without validating it first. Users trust that code examples will work.
- ALWAYS use the v2 API in code examples: URLs must use `https://api.vectara.io/v2/...` and payloads must follow the v2 shape. The `/v1/` API is deprecated. Reject any sub-agent output that uses v1 and rewrite it to v2 before showing the user.
- Vectara REST requests authenticate with the `x-api-key: <YOUR_API_KEY>` header. An `Authorization: Bearer` token is only used with an OAuth client-credentials access token, never with an API key.
- When responding, provide a complete, well-sourced answer with citations.
"""
            }
        ],
            "output_parser": {"type": "default"}
        }
    },
    "tool_configurations": {
        "research_analyst": {
            "type": "sub_agent",
            "description_template": "Research academic research papers for theoretical foundations and research findings.",
            "sub_agent_configuration": {
                "agent_key": research_analyst_key,
                "session_mode": "ephemeral"
            }
        },
        "docs_expert": {
            "type": "sub_agent",
            "description_template": "Research Vectara documentation for API usage, code examples, configuration, and best practices.",
            "sub_agent_configuration": {
                "agent_key": docs_expert_key,
                "session_mode": "ephemeral"
            }
        },
        "web_search_expert": {
            "type": "sub_agent",
            "description_template": "Search the web for up-to-date information, news, and current trends.",
            "sub_agent_configuration": {
                "agent_key": web_search_expert_key,
                "session_mode": "ephemeral"
            }
        },
        "api_validator": {
            "type": "lambda",
            "tool_id": api_validator_id
        }
    }
}

orchestrator_key = delete_and_create_agent(orchestrator_config, "AI Research Orchestrator")

Created agent 'AI Research Orchestrator' (key: agt_ai_research_orchestrator_cb93)


## Step 4: Test the Multi-Agent Workflow

Now let's test the orchestrator with different types of questions to see how it delegates to sub-agents and uses the API validator.

In [8]:
# Helper function to send messages and display responses
def chat_with_agent(agent_key, session_key, message, show_events=False):
    """Send a message to an agent and return the response."""
    message_data = {
        "messages": [
            {
                "type": "text",
                "content": message
            }
        ],
        "stream_response": False
    }
    
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=message_data)
    
    if response.status_code == 201:
        event_data = response.json()
        
        if show_events:
            print("\n------ Agent Events ------")
            for event in event_data.get('events', []):
                event_type = event.get('type', 'INVALID')
                print(f"Event: {event_type}")
                if event_type == 'tool_input':
                    tool_name = event.get('tool_configuration_name', 'N/A')
                    print(f"  Tool: {tool_name}")
                    tool_input = event.get("tool_input", {})
                    if tool_input.get("message"):
                        print(f"  Input: {tool_input['message']}...")
                    # Show API validator inputs
                    if tool_input.get("endpoint"):
                        print(f"  Validating endpoint: {tool_input['endpoint']}")
                if event_type == 'tool_output':
                    tool_name = event.get('tool_configuration_name', 'N/A')
                    print(f"  Tool: {tool_name}")
                    tool_output = event.get("tool_output", {})
                    # Show sub-agent response preview
                    if tool_output.get("sub_agent_response"):
                        print(f"  Response (200 chars): {tool_output['sub_agent_response'][:200]}...")
                    # Show API validator output (a lambda's tool_output is its return value)
                    if "valid" in tool_output:
                        print(f"  Validation Result: {'VALID' if tool_output['valid'] else 'INVALID'}")
                        if tool_output.get("errors"):
                            print(f"    Errors: {tool_output['errors']}")
                        if tool_output.get("warnings"):
                            print(f"    Warnings: {tool_output['warnings']}")
            print("-"*25)
            print("\n")
        
        # Extract agent output
        for event in event_data.get('events', []):
            if event.get('type') == 'agent_output':
                return event.get('content', 'No content')
        
        return "No agent output found"
    else:
        return f"Error: {response.status_code} - {response.text}"

In [9]:
# Create a session for the orchestrator
session_name = f"Orchestrator Session {datetime.now().strftime('%Y%m%d-%H%M%S')}"
session_config = {
    "name": session_name,
    "metadata": {
        "purpose": "sub_agent_demo"
    }
}

response = requests.post(
    f"{BASE_URL}/agents/{orchestrator_key}/sessions",
    headers=headers,
    json=session_config
)
response.raise_for_status()
orchestrator_session_key = response.json()["key"]
print(f"Session Created: {orchestrator_session_key}")

Session Created: ase_orchestrator_session_2026_8cb579499702b1ac419f


### Test 1: Research-Focused Question

This question should primarily use the research_analyst sub-agent:

In [10]:
query = "What are the key innovations in RAG?"
print(f"User: {query}")
print("\n" + "="*80)

response = chat_with_agent(
    orchestrator_key,
    orchestrator_session_key,
    query,
    show_events=True
)

print(f"Agent Response:\n\n{response}")

User: What are the key innovations in RAG?




------ Agent Events ------
Event: input_message
Event: tool_input
  Tool: research_analyst
  Input: Identify the key innovations in Retrieval-Augmented Generation (RAG) as used in natural language processing and AI applications. Focus on recent academic advancements and theoretical innovations that have contributed to the development of RAG....
Event: tool_input
  Tool: web_search_expert
  Input: Search for recent advancements in Retrieval-Augmented Generation (RAG) and identify key innovations in this field, particularly those relevant to real-world applications and industry trends....
Event: tool_output
  Tool: research_analyst
  Response (200 chars): The Retrieval-Augmented Generation (RAG) model, particularly highlighted in the work by Lewis et al. (2020), introduces several key innovations that advance the field of natural language processing (N...
Event: tool_output
  Tool: web_search_expert
  Response (200 chars): Recent advancements in Retrieval-Augmented Generation (RAG) in 2

### Test 2: Implementation Question with API Validation

This question asks for working code, which should trigger the API validator to ensure correctness:

In [11]:
query = "Give me working code to search a corpus for AI research papers using hybrid search with Vectara's API."
print(f"User: {query}")
print("\n" + "="*80)

response = chat_with_agent(
    orchestrator_key,
    orchestrator_session_key,
    query,
    show_events=True
)

print(f"Agent Response:\n\n{response}")

User: Give me working code to search a corpus for AI research papers using hybrid search with Vectara's API.




------ Agent Events ------
Event: input_message
Event: tool_input
  Tool: docs_expert
  Input: Provide guidance on using Vectara's API to perform a hybrid search of a corpus. Include details on constructing a valid JSON request payload for the POST /v2/query endpoint to search for AI research papers....
Event: tool_output
  Tool: docs_expert
  Response (200 chars): To perform a hybrid search using Vectara's API, you'll interact with the `/v2/query` endpoint. This type of search combines semantic search capabilities with traditional keyword-based searching, enhan...
Event: tool_input
  Tool: api_validator
  Validating endpoint: query
Event: tool_output
  Tool: api_validator
  Validation Result: INVALID
    Errors: ["'query' must be the query text as a string, not an object", "Missing required field: 'search'"]
Event: tool_input
  Tool: api_validator
  Validating endpoint: query
Event: tool_output
  Tool: api_validator
Event: tool_input
  Tool: api_validator
  Validating endpoint: query

### Test 3: Current Information Question (Web Search)

This question asks about recent developments that wouldn't be in our indexed research papers or documentation, demonstrating the value of the web search sub-agent:

In [12]:
query = "What are the latest developments in RAG technology in 2025? Are there any new techniques or frameworks that have emerged recently?"
print(f"User: {query}")
print("\n" + "="*80)

response = chat_with_agent(
    orchestrator_key,
    orchestrator_session_key,
    query,
    show_events=True
)

print(f"Agent Response:\n\n{response}")

User: What are the latest developments in RAG technology in 2025? Are there any new techniques or frameworks that have emerged recently?




------ Agent Events ------
Event: input_message
Event: tool_input
  Tool: web_search_expert
  Input: Search for the latest developments in Retrieval-Augmented Generation (RAG) technology as of 2025. Focus on any new techniques, frameworks, or significant advancements that have recently emerged in this field....
Event: tool_output
  Tool: web_search_expert
  Response (200 chars): As of 2025, Retrieval-Augmented Generation (RAG) technology has seen significant advancements, with several new developments in techniques and frameworks:

1. **Security and Scale**: RAG systems, whic...
Event: agent_output
-------------------------


Agent Response:

As of 2025, Retrieval-Augmented Generation (RAG) technology has undergone several significant advancements, including new techniques and frameworks that enhance its efficiency and capabilities:

1. **Security and Scale**: Recent developments involve creating security-hardened and efficiently scalable RAG systems. These systems are explicitly desi